# Data Preparation for Indic Multilingual ASR

Companion notebook to the **Data Preparation** deck. Every step is an NVIDIA
NeMo script or a NeMo configuration field; nothing here reimplements
functionality NeMo already provides.

| Deck slide | Slide title | Notebook section |
|---|---|---|
| — | — | 0. Environment and preflight |
| 2 | A Single Model Across Indic Languages | framing; no section |
| 3 | Publicly Available Indic Speech Corpora | 1. Corpora and manifests |
| 4 | Normalisation and Inverse Text Normalisation | 2. Text normalisation and ITN |
| 5 | Acoustic Robustness Through Augmentation | 3. Noise corpora · 3b. Device and channel · 3c. Codecs |
| 6 | Tokenizer Design Across Indic Scripts | 4. Tokenizer and vocabulary |
| 7 | Randomise Language and Condition, Not Duration | 5. Tarring, weighting and bucketing |
| 8 | Recommended Next Steps | 6. Evaluation sets · 7. Checklist |

Runs on an NVIDIA Brev instance with the NeMo container. Outside that
environment every section that needs NeMo, a GPU or real data skips with an
explicit message rather than failing — see the preflight below.

---
## 0. Environment

### Running this on NVIDIA Brev

Configure a Launchable with:

| Setting | Value |
|---|---|
| Compute | a single GPU instance; set **Disk Storage** generously — corpora are large |
| Container | Single Container, `nvcr.io/nvidia/nemo:<tag>` — pin the tag from the current NGC catalog |
| JupyterLab | select **No** if the container provides its own Jupyter server, to avoid a port conflict |
| Launch parameter | `NGC_API_KEY`, used to pull the container and the base checkpoint |
| Secure Link | port `8888`, which gives JupyterLab a URL behind NVIDIA authentication |

The NeMo container supplies torch, NeMo, Lhotse, `nemo_text_processing`,
SentencePiece and soundfile, so nothing in this notebook installs a framework.

> **One Brev behaviour to know.** In container mode a git repository is cloned
> onto the *instance* at `/home/ubuntu/<repo>`, not inside the container. Mount
> it as a volume or it will not appear in JupyterLab.

The notebook does not name the container image anywhere — it verifies whatever
environment it is running inside, so the same file works on Brev, on a local
container, or on a bare instance with reduced capability.

### Preflight

Run this first. It establishes an `ENV` dictionary that every later section
consults, so cells skip with an explicit message rather than failing deep in
the notebook or silently doing nothing.

In [ ]:
import importlib, shutil, subprocess, os, sys
from pathlib import Path

def _mod(name):
    try:
        importlib.import_module(name); return True
    except Exception:
        return False

def _gpu():
    if not shutil.which('nvidia-smi'):
        return None
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total',
             '--format=csv,noheader'],
            capture_output=True, text=True, check=True).stdout.strip()
        return [l.strip() for l in out.split('\n') if l.strip()]
    except Exception:
        return None

ENV = {
    'nemo'      : _mod('nemo.collections.asr'),
    'nemo_text' : _mod('nemo_text_processing'),
    'torch'     : _mod('torch'),
    'sentencepiece': _mod('sentencepiece'),
    'soundfile' : _mod('soundfile'),
    'matplotlib': _mod('matplotlib'),
    'lhotse'    : _mod('lhotse'),
    'kaldialign': _mod('kaldialign'),
    'ffmpeg'    : bool(shutil.which('ffmpeg')),
    'sox'       : bool(shutil.which('sox')),
}
GPUS = _gpu()
ENV['gpu'] = GPUS is not None

print(f"{'component':<18}{'status'}")
print('-' * 40)
for k, v in ENV.items():
    print(f"{k:<18}{'ok' if v else 'MISSING'}")
print()
for g in (GPUS or ['no GPU visible']):
    print('gpu:', g)

missing = [k for k, v in ENV.items() if not v]
if missing:
    print()
    print('missing:', ', '.join(missing))
    print('Sections depending on these will skip with a message.')

Two packages are absent from the NeMo container and are needed here: `sox`
with its format libraries, used by the built-in codec perturbation, and
`kaldialign`, which the word-error-rate module imports at load time. Install
them once per instance.

In [ ]:
def install_extras():
    cmds = []
    if not ENV['sox']:
        cmds.append('apt-get update -qq && apt-get install -y -qq sox libsox-fmt-all')
    if not ENV['kaldialign']:
        cmds.append(f'{sys.executable} -m pip install -q kaldialign')
    if not cmds:
        print('nothing to install'); return
    for c in cmds:
        print('$', c)
        subprocess.run(c, shell=True, check=False)
    ENV['sox'] = bool(shutil.which('sox'))
    ENV['kaldialign'] = _mod('kaldialign')
    print('sox:', ENV['sox'], ' kaldialign:', ENV['kaldialign'])

install_extras()

In [ ]:
# The NeMo scripts referenced throughout live in the Speech repository.
SPEECH = Path(os.environ.get('SPEECH_REPO', '/workspace/Speech'))

def ensure_speech_repo():
    if SPEECH.exists():
        print('present:', SPEECH); return True
    if not shutil.which('git'):
        print('git unavailable; clone NVIDIA-NeMo/Speech manually'); return False
    SPEECH.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/NVIDIA-NeMo/Speech.git', str(SPEECH)],
                       capture_output=True, text=True)
    print('cloned' if r.returncode == 0 else f'clone failed: {r.stderr[-300:]}')
    return SPEECH.exists()

ENV['speech_repo'] = ensure_speech_repo()

DATA   = Path(os.environ.get('DATA_ROOT',  '/workspace/data'))
MODELS = Path(os.environ.get('MODEL_ROOT', '/workspace/models'))
WORK   = Path(os.environ.get('WORK_ROOT',  '/workspace/work'))
for p in (DATA, MODELS, WORK):
    p.mkdir(parents=True, exist_ok=True)
print('data  :', DATA)
print('models:', MODELS)
print('work  :', WORK)

### A synthetic corpus, so the analysis sections run immediately

Corpus downloads take time and some require registration. This generates a
small synthetic set — frequency sweeps, tones and shaped noise — with a valid
manifest, using only the Python standard library.

It is not speech and will not train anything. Its purpose is that the codec,
spectrogram, bandwidth and tokenizer sections below can be executed and
understood before a single real file arrives. Every one of those sections works
identically on real audio.

In [ ]:
import wave, struct, math, random, json as _json

SR   = 16000     # model sample rate, used throughout
SEED = 42        # fixed so every rendered condition is reproducible
SYN  = WORK / 'synthetic'

def _write_wav(path, samples, sr=SR):
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), 'w') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(b''.join(struct.pack('<h', int(max(-1, min(1, s)) * 26000))
                               for s in samples))

def _sweep(dur=2.0, f0=120, f1=7200, sr=SR):
    n = int(dur * sr)
    return [0.6 * math.sin(2*math.pi*(f0 + (f1-f0)*i/n) * i/sr) for i in range(n)]

def _tone_stack(dur=2.0, fund=180, sr=SR):
    n = int(dur * sr); out = []
    for i in range(n):
        t = i/sr
        v = sum(0.5/(k+1) * math.sin(2*math.pi*fund*(k+1)*t) for k in range(12))
        out.append(0.5 * v * (0.6 + 0.4*math.sin(2*math.pi*3*t)))
    return out

def _shaped_noise(dur=2.0, sr=SR, seed=0):
    rng = random.Random(seed); prev = 0.0; out = []
    for _ in range(int(dur*sr)):
        x = rng.uniform(-1, 1)
        prev = 0.85*prev + 0.15*x        # low-pass shaped, speech-like envelope
        out.append(prev * 1.5)
    return out

def build_synthetic_corpus(n_per_lang=3, langs=('hi', 'mr')):
    rows = []
    for lang in langs:
        for k in range(n_per_lang):
            gen = (_sweep, _tone_stack, _shaped_noise)[k % 3]
            samples = gen(seed=k) if gen is _shaped_noise else gen()
            p = SYN / lang / f'{lang}_{k:03d}.wav'
            _write_wav(p, samples)
            rows.append({
                'audio_filepath': str(p),
                'duration': round(len(samples)/SR, 3),
                'text': f'synthetic utterance {k} in {lang}',
                'lang': lang,
            })
    manifest = SYN / 'synthetic_manifest.json'
    manifest.write_text('\n'.join(_json.dumps(r, ensure_ascii=False) for r in rows) + '\n')
    print(f'{len(rows)} files, {sum(r["duration"] for r in rows):.1f}s total')
    print('manifest:', manifest)
    return manifest, rows

SYN_MANIFEST, SYN_ROWS = build_synthetic_corpus()
REFERENCE_WAV = Path(SYN_ROWS[0]['audio_filepath'])
print('reference file for the sections below:', REFERENCE_WAV)

> Throughout the rest of the notebook, `REFERENCE_WAV` and `SYN_MANIFEST` are
> the synthetic defaults. Point them at real audio and a real manifest and
> nothing else changes.

---
## 1. Corpora and manifests

*Deck slide 3 — Publicly Available Indic Speech Corpora.*

The published Indic corpora let the whole pipeline be validated before any
proprietary audio is collected. Each is converted to the same NeMo manifest
format: newline-delimited JSON, one utterance per line.

In [ ]:
# The reference Indic pipeline ships a preparation script covering the
# published corpora. See bgiddwani-ai/multilingual_nemo_asr (branch: nemotron).
#
#   python data_prep.py --lang <code> --split train \
#     --dataname <corpus> --num_workers 8
#
# Available corpora: indicvoices, kathbath, shrutilipi, lahaja,
#                    svarah (Indian English), springinx, tamil_asr_corpus

CORPORA = ['indicvoices', 'kathbath', 'shrutilipi', 'lahaja', 'svarah']
LANGS   = os.environ.get('TARGET_LANGS', 'hi,mr').split(',')
print('languages in scope:', LANGS)

### Inspect a manifest with NeMo's own utilities

`manifest_utils` reads and writes the manifest format; there is no need to
parse the JSON by hand.

In [ ]:
# NeMo's own manifest utility; the fallback exists only so this notebook
# remains inspectable outside the container.
if ENV['nemo']:
    from nemo.collections.asr.parts.utils.manifest_utils import read_manifest
else:
    def read_manifest(path):
        import json
        return [json.loads(l) for l in open(path) if l.strip()]
    print('NeMo absent - using a stdlib manifest reader')

def summarise(manifest_path):
    rows = read_manifest(manifest_path)
    hours = sum(r['duration'] for r in rows) / 3600
    langs = sorted({r.get('lang', 'unknown') for r in rows})
    print(f'{manifest_path}')
    print(f'  utterances : {len(rows)}')
    print(f'  hours      : {hours:.4f}')
    print(f'  languages  : {langs}')
    return rows

rows = summarise(SYN_MANIFEST)

### Per-language inventory

Report surviving hours per language. This table is the input to every
decision in the training presentation — sampling temperature, curriculum
split, and whether a language is viable at all.

In [ ]:
from collections import defaultdict

def inventory(manifests):
    agg = defaultdict(lambda: {'n': 0, 'sec': 0.0})
    for m in manifests:
        for r in read_manifest(m):
            a = agg[r.get('lang', 'unknown')]
            a['n'] += 1
            a['sec'] += r['duration']
    print(f"{'lang':<8}{'utterances':>12}{'hours':>10}")
    print('-' * 30)
    for lang in sorted(agg):
        a = agg[lang]
        print(f"{lang:<8}{a['n']:>12}{a['sec']/3600:>10.1f}")
    return agg

inv = inventory([SYN_MANIFEST])
print()
print('Point this at the real per-language manifests to get the table that')
print('every decision in the training notebook depends on.')

> **Open item.** NeMo does not ship a speaker-disjoint split processor.
> Where a corpus has no dedicated validation split, `data_prep.py` prints a
> warning to that effect. Decide how splits are produced and record it —
> a speaker appearing in both train and test invalidates every subsequent
> accuracy claim.

---
## 2. Text normalisation and inverse text normalisation

*Deck slide 4 — Normalisation and Inverse Text Normalisation.*

Two directions, and they are easy to confuse:

| Direction | Converts | Used for |
|---|---|---|
| Normalisation | written → spoken | preparing training transcripts |
| Inverse normalisation | spoken → written | formatting model output |

The requirement is that training text, evaluation references and hypotheses
all pass through the same treatment. Where they differ, the reported error
rate reflects formatting rather than accuracy.

In [ ]:
if ENV['nemo_text']:
    from nemo_text_processing.text_normalization.normalize import Normalizer
    from nemo_text_processing.inverse_text_normalization.inverse_normalize import (
        InverseNormalizer)

    LANG = 'hi'
    normalizer     = Normalizer(input_case='cased', lang=LANG)
    inv_normalizer = InverseNormalizer(lang=LANG)

    print('normalise      :', normalizer.normalize('12,500', verbose=False))
    print('inverse        :', inv_normalizer.inverse_normalize(
        'bara hazaar paanch sau', verbose=False))
else:
    print('skipped: nemo_text_processing not available in this environment')

### Confirm coverage before committing to formatted output

Grammar maturity varies by language. Instantiating the class is the direct
way to find out whether a language is supported in the deployed version.

In [ ]:
def check_coverage(langs):
    for lang in langs:
        try:
            InverseNormalizer(lang=lang)
            print(f'{lang:<6} inverse normalisation available')
        except Exception as e:
            print(f'{lang:<6} NOT available  ({type(e).__name__})')

LANGS = ['hi', 'mr', 'ta', 'te', 'kn', 'ml', 'en']
if ENV['nemo_text']:
    check_coverage(LANGS)
else:
    print('skipped: nemo_text_processing not available')

> Where a language is not covered, formatting is either custom grammar work
> or it moves to the application layer. Both are legitimate; the decision
> should be recorded before the first evaluation is reported.
>
> Reference: `NVIDIA/NeMo-text-processing` —
> `tutorials/Text_(Inverse)_Normalization.ipynb` is the entry point and
> `tutorials/WFST_Tutorial.ipynb` covers building grammars.

---
## 3. Noise corpora and augmentation

*Deck slide 5 — Acoustic Robustness Through Augmentation.*

A model trained on clean audio fails under production conditions. Augmentation
falls into four categories, and they are not equally important:

| Category | What it models | Priority |
|---|---|---|
| Additive noise | background floor, interfering speech | the dominant source of degradation |
| Reverberation | room size, distance from the microphone | matters most for speakerphone paths |
| Codec and channel | telephony and VoIP transformation | matters wherever calls are carried |
| Speed and gain | resampling and level variation | not a channel effect; cheap extra diversity |

**Match the production distribution.** Where the channel cannot be guaranteed,
cover the range rather than guessing a single point.

NeMo ships fetchers that download and format standard noise and
room-impulse-response databases into the manifest format.

In [ ]:
# Environmental noise
#   python $SPEECH/scripts/dataset_processing/get_demand_data.py \
#     --data_root /data/noise --data_sets ALL
#
# Room impulse responses and isotropic noise
#   python $SPEECH/scripts/dataset_processing/get_openslr_rir_data.py \
#     --data_root /data/rir

NOISE = DATA.parent / 'noise'
RIR   = DATA.parent / 'rir'
print(NOISE, RIR)

### Split the noise bank by recording session

Training-time and evaluation-time noise must come from disjoint sessions.
Two clips cut from one recording are near-duplicates; sharing them between
train and eval inflates the measured gain.

Plan for source diversity rather than clip count — below roughly ten
distinct sessions per category a model memorises clip spectra instead of
learning robustness.

### The dataloader determines which augmentation path applies

This is the single most common configuration error in this area.

| Dataloader | Augmentation | Notes |
|---|---|---|
| Lhotse (`use_lhotse: true`) | native noise mixing and speed perturbation | keeps prompts, weighting, bucketing |
| Classic | full `augmentor:` perturbation library | reverberation, codec, gain, loudness-normalised mixing |

**The `augmentor:` block is silently discarded when Lhotse or DALI is
enabled.** The configuration still validates and training still runs.

In [ ]:
LHOTSE_AUGMENTATION = '''
model:
  train_ds:
    use_lhotse: true
    noise_path: /data/noise/train_noise.json   # manifest, tarred, cuts or shar
    noise_snr: [5.0, 20.0]
    noise_mix_prob: 0.5
    perturb_speed: true
'''

CLASSIC_AUGMENTATION = '''
model:
  train_ds:
    use_lhotse: false
    augmentor:
      noise_norm:
        prob: 0.5
        manifest_path: /data/noise/train_noise.json
        min_snr_db: 5
        max_snr_db: 20
        norm_to_db: -25.0
      impulse:
        prob: 0.3
        manifest_path: /data/rir/rir.json
      gain:
        prob: 0.3
        min_gain_dbfs: -6
        max_gain_dbfs: 6
'''
print(LHOTSE_AUGMENTATION)

### Verify augmentation is actually running

Two independent checks. Run both before committing GPU time to a long job.

In [ ]:
def assert_augmentation_live(asr_model):
    """Print the constructed augmentation pipeline (classic dataloader)."""
    aug = asr_model._train_dl.dataset.augmentor
    pipeline = [(p, type(o).__name__) for p, o in aug._pipeline]
    if not pipeline:
        raise RuntimeError('augmentation pipeline is empty')
    for prob, name in pipeline:
        print(f'  {name:<32} p={prob}')
    return pipeline

# Second check: an augmented run costs CPU, so it must show measurably
# lower steps/sec than an unaugmented control. Identical throughput means
# augmentation is not running.

> **Expect a trade.** Augmentation raises accuracy on degraded audio and
> lowers it slightly on clean audio. State that before the first checkpoint
> is reviewed, or the dip reads as a regression.
>
> Reference: `NVIDIA-NeMo/Speech` — `tutorials/asr/Online_Noise_Augmentation.ipynb`.

---
## 3b. Device and channel conditions

*Deck slide 5 — Acoustic Robustness Through Augmentation.*

The same sentence reaching the model through a handset held to the ear, a
speakerphone across a room, a messaging app, or a laptop microphone array is
four different signals. The words are identical; the spectrum is not.

There are two ways to obtain these conditions, and they answer different
questions:

| Approach | What it gives | What it cannot give |
|---|---|---|
| **Record** the same script on each device | ground truth, including device-side processing | scale, and controlled comparison |
| **Simulate** the channel with NeMo perturbations | scale, exact repeatability, controlled SNR | on-device noise suppression, beamforming, automatic gain |

Record a small reference set to know what the real conditions look like, then
simulate at scale to train and evaluate against them.

### What is physically happening on each path

| Path | Bandwidth | Codec | Room | Device processing |
|---|---|---|---|---|
| Close-talk reference | wideband | none | negligible | none |
| Mobile handset, held to ear | narrowband | AMR-NB or similar | negligible | noise suppression, AGC |
| Speakerphone | narrowband | AMR-NB or similar | strong reverberation, distance | echo cancellation, AGC |
| PSTN landline | narrowband | G.711 | varies | minimal |
| Messaging or VoIP app | wideband | Opus, typically | varies | heavy noise suppression |
| Laptop microphone array | wideband | none, or app codec | moderate reverberation | multi-microphone beamforming to a single channel, noise suppression, AGC |

Two of these deserve comment because they are commonly misread.

**A laptop array is not simply a distant microphone.** Several microphones are
combined into one channel by beamforming, which suppresses off-axis sound
before the signal ever reaches the application. The result is wideband and
relatively clean, but carries processing artefacts — spectral holes where
suppression acted, and level changes from automatic gain. NeMo perturbations
do not model this. Only recording does.

**A speakerphone is mostly a room, not a codec.** Distance and reverberation
dominate; the codec is secondary. That ordering matters when deciding what to
augment with.

### Recording protocol

Keep it small and controlled. The point is a reference, not a corpus.

- One speaker, one script, roughly thirty seconds, read identically on every device.
- Include domain vocabulary in the script — that is where degradation is felt.
- Record simultaneously where possible, so room and voice are held constant.
- Note the distance, the room, and whether the application had noise
  suppression enabled. Without that, the recordings are not comparable.
- Convert everything to the model's sample rate on arrival and record the
  original rate in the manifest.

In [ ]:
# Real device recordings when available; the synthetic reference otherwise so
# the comparison below can be demonstrated immediately.
REF_DIR = Path(os.environ.get('DEVICE_REF', str(WORK / 'device_reference')))

DEVICE_RECORDINGS = {
    'close_talk'   : REF_DIR / 'close_talk.wav',
    'handset'      : REF_DIR / 'mobile_handset.wav',
    'speakerphone' : REF_DIR / 'mobile_speakerphone.wav',
    'voip_app'     : REF_DIR / 'messaging_app.wav',
    'laptop_array' : REF_DIR / 'laptop_builtin.wav',
}

for name, path in DEVICE_RECORDINGS.items():
    print(f'{name:<14} {"present" if path.exists() else "not recorded yet":<18} {path}')

SOURCE_WAV = (DEVICE_RECORDINGS['close_talk']
              if DEVICE_RECORDINGS['close_talk'].exists() else REFERENCE_WAV)
print()
print('source for the sections below:', SOURCE_WAV)

### Simulating the channel with NeMo perturbations

Each profile below is a composition of perturbations NeMo already provides.
No new perturbation is written.

Three constraints on the codec perturbation, all of which fail quietly rather
than loudly:

- Supported codecs are **`g711`, `amr-nb`, `ogg`**. There is no Opus, GSM,
  MP3 or G.722. Messaging apps typically use Opus, so `ogg` is the nearest
  available proxy — state that rather than implying an exact match.
- `g711` is A-law, the European variant. North American telephony is mu-law.
- It assumes the model sample rate and writes its temporary file at that rate.
  Feeding it already-downsampled audio produces incorrect output silently.

Smoke-test each codec on a one-second file before a long run; `amr-nb`
requires a `sox` build with AMR support that many images lack.

In [ ]:
if ENV['nemo']:
    from nemo.collections.asr.parts.preprocessing.segment import AudioSegment
    from nemo.collections.asr.parts.preprocessing.perturb import (
        TranscodePerturbation, ImpulsePerturbation,
        NoisePerturbationWithNormalization, GainPerturbation)

    def codec_available(codec, probe_wav=None):
        """Confirm a codec works before relying on it in a long run."""
        try:
            seg = AudioSegment.from_file(str(probe_wav or REFERENCE_WAV),
                                        target_sr=SR, offset=0, duration=1.0)
            TranscodePerturbation(codecs=[codec], rng=SEED).perturb(seg)
            return True
        except Exception as e:
            print(f'  {codec}: unavailable ({type(e).__name__})')
            return False

    for c in ('g711', 'amr-nb', 'ogg'):
        print(f'{c:<10}', codec_available(c))
else:
    print('skipped: the built-in perturbations require NeMo')

In [ ]:
if ENV['nemo']:
    # Channel profiles, each built only from NeMo perturbations.
    # rir_manifest and noise_manifest come from the fetchers in section 3.
    
    def build_profiles(rir_manifest, noise_manifest, seed=SEED):
        return {
            'reference': [],
    
            # handset: codec dominates, light ambient noise, no room
            'handset': [
                NoisePerturbationWithNormalization(
                    manifest_path=str(noise_manifest),
                    min_snr_db=18, max_snr_db=25, rng=seed),
                TranscodePerturbation(codecs=['amr-nb'], rng=seed),
            ],
    
            # speakerphone: room dominates, then codec, then level loss
            'speakerphone': [
                ImpulsePerturbation(manifest_path=str(rir_manifest), rng=seed),
                NoisePerturbationWithNormalization(
                    manifest_path=str(noise_manifest),
                    min_snr_db=8, max_snr_db=15, rng=seed),
                GainPerturbation(min_gain_dbfs=-8, max_gain_dbfs=-2, rng=seed),
                TranscodePerturbation(codecs=['amr-nb'], rng=seed),
            ],
    
            # landline: A-law, minimal room
            'pstn_landline': [
                TranscodePerturbation(codecs=['g711'], rng=seed),
            ],
    
            # messaging app: wideband lossy codec, moderate room
            # ogg stands in for Opus; not an exact match
            'voip_app': [
                ImpulsePerturbation(manifest_path=str(rir_manifest), rng=seed),
                TranscodePerturbation(codecs=['ogg'], rng=seed),
            ],
    
            # laptop array: moderate room, no codec.
            # Beamforming and on-device suppression are NOT represented here.
            'laptop_array': [
                ImpulsePerturbation(manifest_path=str(rir_manifest), rng=seed),
                NoisePerturbationWithNormalization(
                    manifest_path=str(noise_manifest),
                    min_snr_db=15, max_snr_db=22, rng=seed),
            ],
        }
    
    def render(source_wav, perturbations, out_wav, sr=SR):
        """Apply a profile in order and write the result to disk."""
        seg = AudioSegment.from_file(str(source_wav), target_sr=sr)
        for p in perturbations:
            p.perturb(seg)
        import soundfile as sf
        Path(out_wav).parent.mkdir(parents=True, exist_ok=True)
        sf.write(str(out_wav), seg.samples, sr)
        return out_wav
else:
    print('skipped: channel profiles require the NeMo perturbations')

### Listen to them

Rendering side by side and listening is the fastest way to build intuition
about which condition is actually hard.

In [ ]:
try:
    from IPython.display import Audio, display, Markdown
    HAVE_IPY = True
except ImportError:
    HAVE_IPY = False
    print('IPython display unavailable outside a notebook kernel')

def audition_profiles(source_wav, profiles, out_dir):
    """Render each channel profile and play them in sequence."""
    if not (ENV['nemo'] and HAVE_IPY):
        print('skipped: needs NeMo and a notebook kernel'); return {}
    out = {}
    for name, perts in profiles.items():
        try:
            out[name] = render(source_wav, perts, Path(out_dir) / f'{name}.wav')
        except Exception as e:
            display(Markdown(f'**{name}** - skipped ({type(e).__name__})')); continue
        display(Markdown(f'**{name}**')); display(Audio(str(out[name])))
    return out

print('call audition_profiles(SOURCE_WAV, PROFILES, WORK/"device_sim") once',
      'noise and RIR manifests exist')

### What the model actually sees

The model does not consume the waveform. It consumes log-mel features, so the
comparison that matters is on the mel spectrogram — computed with the model's
own front-end rather than a generic one, so the parameters match.

In [ ]:
if ENV['nemo'] and ENV['torch']:
    import torch
    from nemo.collections.asr.modules import AudioToMelSpectrogramPreprocessor
    
    def model_preprocessor(asr_model=None, features=128, n_fft=512):
        """Prefer the checkpoint's own front-end; fall back to matching params."""
        if asr_model is not None:
            return asr_model.preprocessor
        return AudioToMelSpectrogramPreprocessor(
            sample_rate=SR, features=features, n_fft=n_fft,
            window_size=0.025, window_stride=0.01)
    
    def log_mel(wav, preprocessor):
        seg = AudioSegment.from_file(str(wav), target_sr=SR)
        sig = torch.tensor(seg.samples, dtype=torch.float32).unsqueeze(0)
        length = torch.tensor([sig.shape[1]])
        with torch.no_grad():
            feats, feat_len = preprocessor(input_signal=sig, length=length)
        return feats[0].cpu().numpy(), int(feat_len[0])
else:
    print('skipped: mel features need NeMo and torch')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def compare_mels(wavs, preprocessor, reference_key='reference'):
    """Top row: log-mel per condition. Bottom row: difference from reference."""
    mels = {}
    for k, w in wavs.items():
        if Path(w).exists():
            mels[k], _ = log_mel(w, preprocessor)
    if reference_key not in mels:
        raise ValueError(f'reference condition {reference_key!r} not rendered')
    ref = mels[reference_key]
    others = [k for k in mels if k != reference_key]

    fig, ax = plt.subplots(2, len(others) + 1, figsize=(4*(len(others)+1), 6),
                           squeeze=False)
    vmin, vmax = ref.min(), ref.max()

    ax[0][0].imshow(ref, origin='lower', aspect='auto', vmin=vmin, vmax=vmax)
    ax[0][0].set_title(reference_key); ax[0][0].set_ylabel('mel bin')
    ax[1][0].axis('off')

    for i, k in enumerate(others, start=1):
        m = mels[k]
        n = min(ref.shape[1], m.shape[1])
        ax[0][i].imshow(m[:, :n], origin='lower', aspect='auto',
                        vmin=vmin, vmax=vmax)
        ax[0][i].set_title(k)
        d = np.abs(m[:, :n] - ref[:, :n])
        ax[1][i].imshow(d, origin='lower', aspect='auto')
        ax[1][i].set_title(f'|{k} - {reference_key}|')
        ax[1][i].set_xlabel('frame')

    plt.tight_layout(); plt.show()
    return mels

if ENV['nemo'] and ENV['torch'] and ENV['matplotlib'] and RENDERED:
    pp = model_preprocessor()
    MELS = compare_mels(RENDERED, pp)
else:
    MELS = {}
    print('skipped: needs NeMo, torch, matplotlib and at least one rendered condition')

**How to read the difference row.** A bright horizontal band across the upper
mel bins is bandwidth loss — the narrowband paths simply have no energy up
there, and every model trained only on wideband audio has never seen that.
Diffuse brightness smeared along the time axis is reverberation. Patchy bright
spots that come and go are codec artefacts and noise suppression.

### Quantify rather than eyeball

Two numbers per condition. Effective bandwidth localises the loss, and the
mean absolute mel difference summarises its magnitude in the space the model
actually operates in.

In [ ]:
def effective_bandwidth(mel, preprocessor, energy_fraction=0.99):
    """Highest mel bin below which the given fraction of energy sits."""
    power = np.exp(mel).mean(axis=1)
    cumulative = np.cumsum(power) / power.sum()
    top_bin = int(np.searchsorted(cumulative, energy_fraction))
    n_bins = mel.shape[0]
    approx_hz = (top_bin / max(n_bins - 1, 1)) * (SR / 2)
    return top_bin, approx_hz

def condition_report(mels, preprocessor, reference_key='reference'):
    ref = mels[reference_key]
    print(f"{'condition':<16}{'top mel bin':>13}{'approx Hz':>12}{'mel delta':>12}")
    print('-' * 53)
    for k, m in mels.items():
        bin_, hz = effective_bandwidth(m, preprocessor)
        n = min(ref.shape[1], m.shape[1])
        delta = float(np.abs(m[:, :n] - ref[:, :n]).mean())
        print(f'{k:<16}{bin_:>13}{hz:>12.0f}{delta:>12.3f}')

if MELS:
    condition_report(MELS, None)
else:
    print('skipped: no mel features computed')

> The Hertz figure is approximate — mel spacing is non-linear, so treat it as
> an indication of where energy stops rather than an exact cutoff. It is
> sufficient to separate narrowband from wideband paths, which is the point.

### Then measure accuracy, not just spectra

The spectrogram shows what changed. Only the error rate shows whether it
matters. Render one evaluation manifest per condition, holding the transcripts
identical, and score them with the same entry point used in section 6.

In [ ]:
# One manifest per condition, identical utterances and identical text.
#
#   python $SPEECH/examples/asr/speech_to_text_eval.py \
#     model_path=<checkpoint>.nemo \
#     dataset_manifest=/data/device_sim/<condition>_manifest.json \
#     text_processing.do_lowercase=true \
#     text_processing.rm_punctuation=true

print('Report clean, the degraded average, and the worst single condition',
      'together. For traffic where nearly every call is degraded, the',
      'trade favours augmentation heavily.')

> **Ordering to expect.** Additive noise costs substantially more accuracy
> than codec transformation — on published reference measurements the codec
> round trip is a few percent relative, while low signal-to-noise conditions
> run to tens of percent. If a telephony corpus degrades far more than a few
> percent, the codec is not the dominant cause: look to background noise,
> handset and room effects, or domain mismatch.
>
> **What this section cannot tell you.** On-device beamforming, noise
> suppression and automatic gain are applied before the audio reaches any
> pipeline, and are not represented by these perturbations. If laptop or
> messaging-app traffic is a significant share of production, record it.

---
## 3c. Telephony and VoIP codecs

*Deck slide 5 — Acoustic Robustness Through Augmentation.*

A codec is chosen per call by negotiation, so a single deployment sees several.
The damage is done at encode time: the encoder discards information at its own
sample rate and bit rate, and decoding back to the model's rate does not
restore it. What reaches the model is a round trip.

### Which codecs are worth simulating

Ranked by how often they appear in production speech traffic.

| Priority | Codec | Rate | Where it appears |
|---|---|---|---|
| Essential | G.711 A-law | 8 kHz | PSTN and SIP across Europe, India, most of Asia |
| Essential | G.711 μ-law | 8 kHz | PSTN and SIP across North America and Japan |
| Essential | AMR-WB | 16 kHz | VoLTE and HD Voice — the mobile default on modern networks |
| Essential | AMR-NB | 8 kHz | 2G and 3G mobile voice, and VoLTE fallback |
| Essential | Opus | 8–48 kHz | WebRTC, browser calling, messaging applications |
| Useful | G.722 | 16 kHz | enterprise wideband IP handsets and conferencing |
| Useful | G.726 | 8 kHz | ADPCM on legacy trunks and PBX interconnects |
| Low | GSM-FR, Speex | 8–32 kHz | legacy, largely displaced by Opus |

**Two codecs are deliberately not in this notebook.** G.729 is common in
enterprise VoIP but FFmpeg provides a decoder only, and EVS encoders are not
generally available. Neither can be simulated with this toolchain, so where
they are in use those conditions have to be captured by recording real calls.
Everything below is restricted to codecs that actually encode.

### Do not guess the distribution — read it

Codec selection is negotiated per call, and the session border controller or
call platform records which codec was actually used. That log gives the real
distribution for this deployment, which is worth far more than a general
ranking.

Ask the telephony team for the codec breakdown across a representative week,
split by inbound and outbound, mobile and landline. Simulate in proportion to
what that shows.

### A-law and μ-law are not interchangeable

Both are G.711 at 8 kHz and 64 kbps, and they differ only in the companding
curve — but they are different transformations and produce different artefacts.

Two specifics worth carrying into the design:

- NeMo's built-in codec perturbation exposes `g711`, which is **A-law**.
- NVIDIA's published telephony degradation reference for its ASR models is
  measured on a **μ-law** round trip.

For an Indian deployment A-law is the correct PSTN variant, so the built-in
perturbation matches one real condition. It does not match the published
benchmark, and it does not cover mobile or application traffic at all.

| Requirement | Path |
|---|---|
| A-law, AMR-NB, or an Opus-in-Ogg approximation | built-in `transcode_aug` |
| μ-law, AMR-WB, G.722, G.726, GSM, exact Opus | register an additional perturbation |

`register_perturbation()` is the documented extension point, and once registered
a codec is reachable from every `augmentor:` block in the stack. That is the
sanctioned route rather than a workaround.

### Probe the build before relying on any encoder

Codec availability is a property of how FFmpeg was compiled, not of the format.
Several of the encoders above are frequently absent.

In [ ]:
import subprocess, shutil

# codec -> (ffmpeg encoder, native sample rate, extra encode args, container)
CODECS = {
    'g711_alaw' : ('pcm_alaw',            8000,  [],                    'wav'),
    'g711_ulaw' : ('pcm_mulaw',           8000,  [],                    'wav'),
    'g722'      : ('g722',               16000,  [],                    'wav'),
    'g726_32k'  : ('g726',                8000,  ['-b:a', '32k'],       'wav'),
    'gsm_fr'    : ('libgsm',              8000,  [],                    'wav'),
    'amr_nb'    : ('libopencore_amrnb',   8000,  ['-b:a', '12.2k'],     'amr'),
    'amr_wb'    : ('libvo_amrwbenc',     16000,  ['-b:a', '15.85k'],    'amr'),
    'opus_24k'  : ('libopus',            16000,  ['-b:a', '24k'],       'ogg'),
    'speex'     : ('libspeex',            8000,  [],                    'ogg'),
}

def available_encoders():
    if not shutil.which('ffmpeg'):
        print('ffmpeg not on PATH'); return set()
    out = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'],
                         capture_output=True, text=True).stdout
    have = set()
    print(f"{'codec':<12}{'encoder':<22}{'status'}")
    print('-' * 46)
    for name, (enc, *_ ) in CODECS.items():
        ok = f' {enc} ' in out or f' {enc}\n' in out
        if ok: have.add(name)
        print(f"{name:<12}{enc:<22}{'available' if ok else 'MISSING'}")
    print()
    print('g729       (decoder only in ffmpeg)  cannot be simulated')
    print('evs        (no general encoder)       cannot be simulated')
    return have

HAVE = available_encoders() if ENV['ffmpeg'] else set()

> **Observed on a standard image.** Running the probe above on a general-purpose
> container gave the following, which is representative of what to expect:
>
> | Encoder | Status |
> |---|---|
> | `pcm_mulaw`, `pcm_alaw` | available |
> | `g722`, `g726` | available |
> | `libgsm`, `libspeex` | available |
> | `libopus` | available |
> | `libopencore_amrnb` | **missing** |
> | `libvo_amrwbenc` | **missing** |
> | G.729 encoder | not provided by FFmpeg at all |
>
> Both AMR encoders were absent, and AMR-WB is the mobile default on VoLTE —
> which is likely the single largest share of traffic in an Indian deployment.
> Getting AMR requires an FFmpeg built with `libopencore-amrnb` and
> `libvo-amrwbenc`, or the AMR support in a `sox` build for the NeMo built-in
> path. Confirm this before planning around it, because the round trip will
> otherwise fail partway through a long run rather than at configuration time.

### Register a codec perturbation

One class, registered once, covering every encoder the build provides. This
uses NeMo's extension API rather than replacing any part of the augmentation
system, and the result composes with the perturbations already configured.

Two implementations follow, for two different jobs.

The first is a plain round trip used for analysis and listening — it needs only
FFmpeg, so it runs before NeMo is available. The second wraps the same
transformation as a NeMo perturbation so it can be declared in a training
configuration.

In [ ]:
import tempfile

def codec_round_trip(src_wav, codec_name, out_wav, model_sr=SR):
    """Encode at the codec's native rate, decode back. FFmpeg only."""
    enc, native_sr, extra, container = CODECS[codec_name]
    out_wav = Path(out_wav); out_wav.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        mid = Path(tmp) / f'mid.{container}'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(src_wav),
                        '-ar', str(native_sr), '-ac', '1',
                        '-c:a', enc, *extra, str(mid)], check=True)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(mid),
                        '-ar', str(model_sr), '-ac', '1',
                        '-c:a', 'pcm_s16le', str(out_wav)], check=True)
    return out_wav

def render_all(src_wav, codec_names, out_dir):
    done = {'reference': Path(src_wav)}
    for name in codec_names:
        if name not in HAVE:
            print(f'{name:<14} skipped - encoder not in this FFmpeg build'); continue
        try:
            done[name] = codec_round_trip(src_wav, name, Path(out_dir)/f'{name}.wav')
            print(f'{name:<14} rendered')
        except Exception as e:
            print(f'{name:<14} failed: {type(e).__name__}')
    return done

SHORTLIST = ['g711_alaw', 'g711_ulaw', 'g722', 'g726_32k', 'opus_16k', 'amr_wb']
RENDERED = render_all(SOURCE_WAV, SHORTLIST, WORK / 'codec_sim') if ENV['ffmpeg'] else {}

In [ ]:
# Registers the same round trip as a NeMo perturbation, so it can be declared
# in a training configuration. Analysis above needs only codec_round_trip().
if ENV['nemo']:
    from nemo.collections.asr.parts.preprocessing import perturb
    from nemo.collections.asr.parts.preprocessing.perturb import Perturbation
    from nemo.collections.asr.parts.preprocessing.segment import AudioSegment
    import soundfile as sf

    class TelephonyCodecPerturbation(Perturbation):
        """Encode to a telephony codec at its native rate, then decode back."""

        def __init__(self, codecs=('g711_ulaw',), model_sr=SR, rng=None):
            self._codecs = list(codecs)
            self._model_sr = model_sr
            self._rng = rng

        def perturb(self, data, **kwargs):
            import random, tempfile
            rng = random.Random(self._rng) if self._rng is not None else random
            name = rng.choice(self._codecs)
            with tempfile.TemporaryDirectory() as tmp:
                src = Path(tmp) / 'src.wav'
                dst = Path(tmp) / 'dst.wav'
                sf.write(str(src), data._samples, self._model_sr)
                codec_round_trip(src, name, dst, model_sr=self._model_sr)
                out = AudioSegment.from_file(str(dst), target_sr=self._model_sr)
            n = min(len(out.samples), len(data._samples))
            data._samples = out.samples[:n]

    perturb.register_perturbation(name='telephony_codec',
                                  perturbation=TelephonyCodecPerturbation)
    print('registered as telephony_codec - now usable in any augmentor block')
else:
    print('skipped: registration requires NeMo. codec_round_trip() above still works.')

Once registered, the codec is available declaratively alongside the built-in
perturbations, with no further code:

In [ ]:
CODEC_AUGMENTATION = '''
model:
  train_ds:
    use_lhotse: false          # augmentor requires the classic dataloader
    augmentor:
      telephony_codec:
        prob: 0.4
        codecs: [g711_alaw, g711_ulaw, amr_nb, amr_wb, opus_24k]
      noise_norm:
        prob: 0.5
        manifest_path: /data/noise/train_noise.json
        min_snr_db: 5
        max_snr_db: 20
'''
print(CODEC_AUGMENTATION)
print('Mix codecs in proportion to the distribution from the call platform.')

### Bit rate matters as much as codec identity

For the variable-rate codecs the operating point dominates. AMR-NB spans
4.75 to 12.2 kbps and Opus is routinely deployed anywhere from 8 to 32 kbps
for speech. A single label such as "AMR" or "Opus" does not describe a
condition; the rate does.

In [ ]:
BITRATE_SWEEP = {
    'amr_nb_4k75'  : ('libopencore_amrnb',  8000, ['-b:a', '4.75k'], 'amr'),
    'amr_nb_12k2'  : ('libopencore_amrnb',  8000, ['-b:a', '12.2k'], 'amr'),
    'amr_wb_6k6'   : ('libvo_amrwbenc',    16000, ['-b:a', '6.6k'],  'amr'),
    'amr_wb_23k85' : ('libvo_amrwbenc',    16000, ['-b:a', '23.85k'],'amr'),
    'opus_8k'      : ('libopus',           16000, ['-b:a', '8k'],    'ogg'),
    'opus_16k'     : ('libopus',           16000, ['-b:a', '16k'],   'ogg'),
    'opus_32k'     : ('libopus',           16000, ['-b:a', '32k'],   'ogg'),
}
CODECS.update(BITRATE_SWEEP)
print('sweep points added:', list(BITRATE_SWEEP))

### Render, listen, and compare

Reuses the rendering and mel-comparison helpers from section 3b, so the codec
conditions sit on the same axes as the device conditions.

In [ ]:
try:
    from IPython.display import Audio, display, Markdown
    HAVE_IPY = True
except ImportError:
    HAVE_IPY = False

def audition(rendered):
    """Play every rendered condition in sequence."""
    if not HAVE_IPY:
        for name, path in rendered.items():
            print(f'{name:<14} {path}')
        return
    for name, path in rendered.items():
        display(Markdown(f'**{name}**')); display(Audio(str(path)))

if RENDERED:
    audition(RENDERED)
else:
    print('nothing rendered to audition')

In [ ]:
# Same mel comparison used for the device conditions.
# pp = model_preprocessor()
# wavs = {'reference': DEVICE_RECORDINGS['close_talk'], **rendered}
# mels = compare_mels(wavs, pp)
# condition_report(mels, pp)

print('What to look for:')
print('  8 kHz codecs  - energy stops abruptly around the mel bin')
print('                  corresponding to 4 kHz; everything above is empty')
print('  16 kHz codecs - upper bins retained, but quantisation noise appears')
print('  low bit rate  - fine harmonic structure is smoothed away, which')
print('                  is where consonant confusions come from')

### Then measure accuracy per codec

The spectrogram shows what was removed. Only the error rate shows what it
costs. Render one evaluation manifest per codec against identical transcripts
and score them all with the same entry point.

In [ ]:
# python $SPEECH/examples/asr/speech_to_text_eval.py \
#   model_path=<checkpoint>.nemo \
#   dataset_manifest=/data/codec_sim/<codec>_manifest.json \
#   text_processing.do_lowercase=true \
#   text_processing.rm_punctuation=true

print('Report the reference, each codec, and the worst case together.')

**Expected ordering, to be confirmed on your own data.**

Bandwidth is the dominant factor, then bit rate, then the companding or
quantisation scheme:

1. Wideband codecs at generous bit rates — G.722, AMR-WB at high rates, Opus
   at 24 kbps and above — cost relatively little.
2. The 8 kHz family — both G.711 variants, G.726, GSM-FR — cost more, and
   similarly to one another, because the bandwidth loss dominates the
   difference between their coding schemes.
3. Low-rate variable codecs — AMR-NB at 4.75 kbps, Opus at 8 kbps — cost the
   most, because bandwidth loss and aggressive quantisation compound.

On NVIDIA's published reference the G.711 round trip alone is a few percent
relative, whereas low signal-to-noise conditions run to tens of percent. Codec
handling matters, but noise matters more — if a telephony corpus degrades far
beyond a few percent relative, the codec is not the dominant cause.

> **Practical consequence.** Augment with the codecs the call platform
> actually negotiates, weighted by their real share, and evaluate on each
> separately. A single blended telephony number hides the one path that is
> failing.

---
## 4. Tokenizer and vocabulary

*Deck slide 6 — Tokenizer Design Across Indic Scripts.*

One unified tokenizer across all target languages. Excessive sub-word
fragmentation lengthens target sequences, makes alignment harder, and is a
primary cause of reduced accuracy in lower-resource languages.

In [ ]:
# Build the tokenizer from the training manifests of every target language.
#
#   python $SPEECH/scripts/tokenizers/process_asr_text_tokenizer.py \
#     --manifest '<comma-separated manifests>' \
#     --data_root models/indic_tokenizer \
#     --vocab_size 2048 \
#     --tokenizer spe --spe_type bpe \
#     --spe_character_coverage 0.9995

VOCAB_SIZES = [1024, 2048, 4096]
print('sweep these, do not pick one:', VOCAB_SIZES)

### Measure fragmentation, not vocabulary size

The number that matters is tokens per word, per language. Character error
rate responds to fragmentation before word error rate does.

In [ ]:
if ENV['sentencepiece']:
    import sentencepiece as spm

    def fragmentation(model_file, manifests_by_lang, sample=2000):
        sp = spm.SentencePieceProcessor(model_file=model_file)
        print(f"{'lang':<8}{'tokens/word':>14}{'unk rate':>12}")
        print('-' * 34)
        out = {}
        for lang, manifest in manifests_by_lang.items():
            rows = read_manifest(manifest)[:sample]
            toks = words = unk = 0
            for r in rows:
                ids = sp.encode(r['text'], out_type=int)
                toks  += len(ids)
                words += len(r['text'].split())
                unk   += sum(1 for i in ids if i == sp.unk_id())
            out[lang] = toks / max(words, 1)
            print(f'{lang:<8}{out[lang]:>14.2f}{unk/max(toks,1):>12.4f}')
        return out

    print('fragmentation() ready - pass a built tokenizer.model and the',
          'per-language manifests')
else:
    print('skipped: sentencepiece not available')

**Reading the result.** A language whose tokens-per-word is markedly higher
than the others is under-served by the merge table. Rebalance the text pool
or raise the vocabulary size.

A non-zero unknown rate means `spe_character_coverage` is dropping characters
that language actually needs. Raise it.

**Keep the mixture.** If the tokenizer training text is exclusively Indic
script, embedded English fragments heavily at inference. The code-mixed form
must be present in the text used to build the tokenizer.

---
## 5. Tarring, weighting and bucketing

*Deck slide 7 — Randomise Language and Condition, Not Duration.*

Tar shards are read sequentially, so what is written together is seen
together. Shuffle at build time, and finish normalisation, filtering and any
offline rendering **before** this step — the shards are immutable afterwards.

In [ ]:
# One tarred dataset per language.
#
#   python $SPEECH/scripts/speech_recognition/convert_to_tarred_audio_dataset.py \
#     --manifest_path='<lang>_manifest.json' \
#     --target_dir='<lang>_tarred' \
#     --num_shards=256 \
#     --max_duration=30.0 --min_duration=0.025 \
#     --shuffle \
#     --workers=16

print('--shuffle is not optional: without it, batch composition follows',
      'the order the manifest happened to be written in.')

### Combine languages through weighted sampling

Language diversity within a batch comes from the sampler, not from the
shards — each language is stored separately and mixed at this level.
Noise conditions rendered offline should likewise be registered as their own
sources so the sampler mixes them too.

In [ ]:
INPUT_CFG = '''
- type: nemo_tarred
  manifest_filepath: /data/dataset/<lang>_tarred/sharded_manifests/manifest__OP_0..255_CL_.json
  tarred_audio_filepaths: /data/dataset/<lang>_tarred/audio__OP_0..255_CL_.tar
  weight: 1.0
  tags:
    lang: <lang>
    prompt_mode: unified        # see the training notebook
'''
print(INPUT_CFG)

### Duration bucketing

Bucketing groups comparable durations deliberately, to avoid padding waste.
A batch is meant to be length-homogeneous; diversity in length happens across
steps. `synced_randomized` keeps every distributed rank on the same bucket at
the same step.

In [ ]:
# Estimate duration bins from the combined configuration
#   python $SPEECH/scripts/speech_recognition/estimate_duration_bins.py \
#     -b 20 /data/dataset/input_cfg.yaml
#
# Optionally fit batch sizes to available memory
#   python $SPEECH/scripts/speech_recognition/oomptimizer.py \
#     --config-path <training config> \
#     --module-name nemo.collections.asr.models.EncDecRNNTBPEModel \
#     --memory-fraction 0.9 --buckets '<bins from the previous step>'

---
## 6. Evaluation sets

*Deck slide 8 — Recommended Next Steps, third action.*

Evaluation audio must be rendered to disk so that every checkpoint is scored
on byte-identical files. Build one manifest per acoustic condition with the
signal-to-noise ratio held at an exact value and the seed fixed.

In [ ]:
if ENV['nemo']:
    from nemo.collections.asr.parts.preprocessing.perturb import (
        NoisePerturbationWithNormalization)
    
    SNR_LADDER = [20, 10, 5, 0]   # one evaluation manifest per condition
    SEED = 42
    
    def condition_perturber(snr, eval_noise_manifest):
        """min == max fixes the condition at an exact SNR."""
        return NoisePerturbationWithNormalization(
            manifest_path=str(eval_noise_manifest),
            min_snr_db=snr, max_snr_db=snr, rng=SEED)
    
    print('conditions:', SNR_LADDER)
else:
    print('skipped: offline rendering uses the NeMo perturbations')
    SNR_LADDER = [20, 10, 5, 0]
    print('condition ladder:', SNR_LADDER)

Two rules decide whether the comparison is valid:

1. **Hold out the noise.** Evaluation noise comes from recording sessions
   that never appear in training.
2. **Crop the noise to the utterance before mixing**, so the requested and
   realised signal-to-noise ratio agree.

Measure the realised ratio back out of the rendered audio and record it per
file rather than trusting the requested value.

In [ ]:
# Score a checkpoint across the condition matrix with NeMo's own entry point.
#
#   python $SPEECH/examples/asr/speech_to_text_eval.py \
#     model_path=<checkpoint>.nemo \
#     dataset_manifest=<condition>_manifest.json \
#     text_processing.do_lowercase=true \
#     text_processing.rm_punctuation=true

print('The scorer applies no normalisation of its own. Apply identical',
      'text processing to every condition.')

---
## 7. Pre-training checklist

*Deck slide 8 — Recommended Next Steps.*

Run through this before building tar shards, because several items cannot be
corrected afterwards without rebuilding the dataset.

In [ ]:
CHECKLIST = [
    'Per-language surviving hours recorded after filtering, not before',
    'Split policy decided and speaker overlap between train and test ruled out',
    'Normalisation form chosen and applied to every manifest',
    'Inverse normalisation coverage confirmed per target language',
    'Noise bank split by recording session into train and evaluation halves',
    'Augmentation path matched to the dataloader in use',
    'Augmentation pipeline asserted live on a short run',
    'Tokenizer swept and fragmentation reported per language',
    'Code-mixed text present in the tokenizer training pool',
    'Tarring performed with shuffle enabled',
    'Evaluation manifests rendered at fixed SNR with a fixed seed',
]
for i, item in enumerate(CHECKLIST, 1):
    print(f'{i:>3}. [ ] {item}')

### References

| Repository | Contents |
|---|---|
| `NVIDIA-NeMo/Speech` | dataset processing, tokenizers, tarring, bucketing, evaluation |
| `NVIDIA/NeMo-text-processing` | normalisation and inverse normalisation grammars |
| `bgiddwani-ai/multilingual_nemo_asr` | Indic corpus preparation and reference configuration |

Continue with **`02_Model_Training.ipynb`**.